In [31]:
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    hamming_loss
)
from sklearn.metrics import f1_score, hamming_loss, precision_score
import numpy as np
from dotenv import load_dotenv
import os
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from typing import List

load_dotenv()

True

In [32]:
df = pd.read_csv('dataset.csv')

In [33]:
df.head()

,courseId,courseName,courseDescription,courseHours,comp_id,comp_name,comp_description,cat_id,cat_name,tax_id,tax_name
0,c3dc5a27-f76b-46a3-b9c9-644d35a66baf,A arte de falar em público,"Neste curso, você terá a oportunidade de conhe...",20,6461c25c-1491-4610-8ee0-2029d77fa61d,Comunicação e colaboração digital,"Incorporar atividades, tarefas e avaliações de...",05ac4a5c-019d-4ec7-83cc-12ca141322fa,Promoção da Competência Digital dos(as) Estuda...,85f646ca-67c5-439c-8cc1-df4fed9e0ae7,Lembrar
1,c3dc5a27-f76b-46a3-b9c9-644d35a66baf,A arte de falar em público,"Neste curso, você terá a oportunidade de conhe...",20,6461c25c-1491-4610-8ee0-2029d77fa61d,Comunicação e colaboração digital,"Incorporar atividades, tarefas e avaliações de...",05ac4a5c-019d-4ec7-83cc-12ca141322fa,Promoção da Competência Digital dos(as) Estuda...,6152c8bb-1632-42f0-9599-919b6bf15ecb,Entender
2,c3dc5a27-f76b-46a3-b9c9-644d35a66baf,A arte de falar em público,"Neste curso, você terá a oportunidade de conhe...",20,6461c25c-1491-4610-8ee0-2029d77fa61d,Comunicação e colaboração digital,"Incorporar atividades, tarefas e avaliações de...",05ac4a5c-019d-4ec7-83cc-12ca141322fa,Promoção da Competência Digital dos(as) Estuda...,7b16819f-cb8a-406d-9ab9-a335a50143cb,Aplicar
3,c3dc5a27-f76b-46a3-b9c9-644d35a66baf,A arte de falar em público,"Neste curso, você terá a oportunidade de conhe...",20,cdbece75-961d-4176-b34b-66e1a530dcf2,Comunicação institucional,Usar tecnologias digitais para melhorar a comu...,690c731c-31f8-46f6-94f6-94d952223a08,Envolvimento Profissional,85f646ca-67c5-439c-8cc1-df4fed9e0ae7,Lembrar
4,c3dc5a27-f76b-46a3-b9c9-644d35a66baf,A arte de falar em público,"Neste curso, você terá a oportunidade de conhe...",20,cdbece75-961d-4176-b34b-66e1a530dcf2,Comunicação institucional,Usar tecnologias digitais para melhorar a comu...,690c731c-31f8-46f6-94f6-94d952223a08,Envolvimento Profissional,6152c8bb-1632-42f0-9599-919b6bf15ecb,Entender


In [34]:
df_courses = (
    df[["courseId", "courseName", "courseDescription"]]
    .drop_duplicates(subset=["courseId"])
    .reset_index(drop=True)
)

In [35]:
df_competencies = (
    df[["comp_id", "comp_name", "comp_description", "cat_id", "cat_name"]]
    .drop_duplicates(subset=["comp_id"])
    .reset_index(drop=True)
)

In [36]:
llm = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0.0,
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [37]:
llm.model_name

'gpt-5-mini'

In [38]:
df_competencies.head()

,comp_id,comp_name,comp_description,cat_id,cat_name
0,6461c25c-1491-4610-8ee0-2029d77fa61d,Comunicação e colaboração digital,"Incorporar atividades, tarefas e avaliações de...",05ac4a5c-019d-4ec7-83cc-12ca141322fa,Promoção da Competência Digital dos(as) Estuda...
1,cdbece75-961d-4176-b34b-66e1a530dcf2,Comunicação institucional,Usar tecnologias digitais para melhorar a comu...,690c731c-31f8-46f6-94f6-94d952223a08,Envolvimento Profissional
2,3418004f-90fd-4a64-bb7c-fab6c1f8f6cc,Comunicação interpessoal,Iniciar e manter comunicações respeitosas e co...,514d5868-5f15-4ce9-b74a-d14f5310b54e,Gestão de Relacionamentos
3,e4f31519-2fad-4fe8-b476-5da4715a16e9,Ensino,Planificar e implementar dispositivos e recurs...,cac54f5d-834c-499c-bf8d-e549fa3828af,Ensino e Aprendizagem
4,6815ce74-37f5-40d0-ac92-285d0c126ee9,Promoção da Inclusão e Acessibilidade,"Planejar, implementar e/ou monitorar políticas...",27ccdc3f-64d6-4748-960b-171e9c804d23,Transversais


In [39]:
df_courses.head()

,courseId,courseName,courseDescription
0,c3dc5a27-f76b-46a3-b9c9-644d35a66baf,A arte de falar em público,"Neste curso, você terá a oportunidade de conhe..."
1,8153e4a8-c7d2-43b1-a447-78a42fc2c5f7,Abordagens Pedagógicas Modernas na Educação a ...,O curso apresenta e discute abordagens pedagóg...
2,c17b8522-18b7-4d5d-9c4d-7e78685792d8,Acessibilidade em espaços de uso público no Br...,O curso tem como foco a identificação dos prob...
3,8dbe8524-6fbe-4a38-8311-f3d99f988f78,Acessibilidade em Processos Seletivos Discentes,Formação oferecida gratuitamente pelo Institut...
4,8c9c4ffe-9f34-4111-ab38-14c77684abd8,Acessibilidade e Tecnologia,O curso tem como objetivos apresentar os conce...


In [40]:
df_true = (
    df[["courseId", "comp_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
df_true["real"] = 1 

In [41]:
texts = []
metadatas = []

for _, row in df_competencies.iterrows():
    text = (
        f"ID: {row['comp_id']}\n"
        f"Name: {row['comp_name']}\n"
        f"Description: {row['comp_description']}"
    )
    texts.append(text)
    metadatas.append({
        "comp_id": row["comp_id"],
        "comp_name": row["comp_name"],
    })

vectorstore = FAISS.from_texts(
    texts=texts,
    embedding=embeddings,
    metadatas=metadatas,
)

### 1. General formula

$$
k = \mu + 2\sigma
$$

### Substituting the approximate values

$$
\mu \approx 2.80117, \qquad \sigma \approx 2.14878
$$

### Step-by-step calculation

$$
2\sigma = 2 \times 2.14878 = 4.29756
$$

$$
k = \mu + 2\sigma = 2.80117 + 4.29756 = 7.09873
$$

### Therefore

$$
k \approx 7.1 \;\;\Rightarrow\;\; \text{practical range} \approx 7\text{–}8
$$

An approximate interval of two standard deviations around the mean, $\mu \pm 2\sigma$, was adopted as a practical estimate of the region where most courses are concentrated. Although the distribution is not perfectly normal, the empirical rule suggests that values within this interval represent approximately 95% of the observations.

In [ ]:
top_k = 7

In [43]:
retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})

In [44]:
class CourseLabel(BaseModel):
    course_id: str = Field(..., description="Course ID (courseId)")
    selected_competencies: List[str] = Field(
        ..., description="List of selected competency IDs (comp_id)"
    )

parser = JsonOutputParser(pydantic_object=CourseLabel)

In [45]:
SYSTEM_PROMPT_TEMPLATE = """
You are an expert in mapping courses to competencies.

All course titles, descriptions, and competencies are written in Brazilian Portuguese.
DO NOT TRANSLATE, REWRITE, OR MODIFY ANY OF THESE TEXTS. USE THEM ONLY AS EVIDENCE.

Task:
Given a course description and a list of candidate competencies, select the top {top_k}
competencies that are explicitly related to the course content.

Instructions:
- CONSIDER ONLY the candidate competencies provided.
- USE ONLY the course title and description as evidence.
- Evaluate each competency based on its name and description (in Portuguese).
- Select a competency ONLY IF THERE IS A CLEAR AND EXPLICIT CONNECTION to the course.
- RETURN EXACTLY {top_k} COMPETENCIES, unless fewer than {top_k} are relevant.
- If none apply, RETURN AN EMPTY LIST.

Rules:
- DO NOT REWRITE, TRANSLATE, OR MODIFY ANY COMPETENCY TEXT.
- DO NOT HALLUCINATE NEW COMPETENCIES.
- BASE ALL DECISIONS STRICTLY ON THE PROVIDED TEXT.

RETURN ONLY THE JSON OBJECT DEFINED BY THE SCHEMA.
"""

SYSTEM_PROMPT = SYSTEM_PROMPT_TEMPLATE.format(top_k=top_k)

In [46]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        (
            "human",
            """Course (ID: {course_id})

Title: {course_title}
Description: {course_description}

Candidate competencies (ID - Name: Description):
{candidate_competencies}

{format_instructions}
""",
        ),
    ]
)

chain = prompt | llm | parser

In [47]:
def build_candidate_text(docs):
    lines = []
    for d in docs:
        comp_id = d.metadata["comp_id"]
        comp_name = d.metadata["comp_name"]
        lines.append(
            f"{comp_id} - {comp_name}: {d.page_content}"
        )
    return "\n".join(lines)


def label_course(course_row):
    course_id = str(course_row["courseId"])
    course_name = str(course_row["courseName"])
    course_description = str(course_row["courseDescription"])

    course_text = f"{course_name}: {course_description}"

    # 1) Retrieve candidate competencies by similarity
    candidates = retriever.invoke(course_text)

    # 2) Invoke LLM + parser
    result = chain.invoke(
        {
            "course_id": course_id,
            "course_title": course_name,
            "course_description": course_description,
            "candidate_competencies": build_candidate_text(candidates),
            "format_instructions": parser.get_format_instructions(),
        }
    )

    # result is a dict, so we access by key
    course_id_out = str(result["course_id"])
    competencies = result["selected_competencies"]

    return {
        "courseId": course_id_out,
        "selected_competencies": competencies,
    }

In [ ]:
# -------------------------------------------------------------------
# 1) Run the LLM for all courses and build the predictions dataset
# -------------------------------------------------------------------

results = []
for _, row in df_courses.iterrows():
    r = label_course(row)
    results.append(r)

# One row per course, each containing a list of competencies
df_pred = pd.DataFrame(results)

# Ensure the field is always a list (in case it comes as None or string)
df_pred["selected_competencies"] = df_pred["selected_competencies"].apply(
    lambda x: x if isinstance(x, list) else []
)

In [ ]:
# -------------------------------------------------------------
# 2) Explode the lists to obtain one row per (courseId, comp_id)
# -------------------------------------------------------------

df_pred_long = df_pred.explode("selected_competencies").rename(
    columns={"selected_competencies": "comp_id"}
)

# Remove rows where no competency exists (empty lists become NaN after explode)
df_pred_long = df_pred_long.dropna(subset=["comp_id"])

# Ensure string types to match during the merge
df_pred_long["courseId"] = df_pred_long["courseId"].astype(str)
df_pred_long["comp_id"] = df_pred_long["comp_id"].astype(str)

# Prediction flag: 1 = the model selected this competency for the course
df_pred_long["pred"] = 1

In [ ]:
# -------------------------------------------------------------
# 3) Prepare ground-truth labels and compare (true vs predicted)
# -------------------------------------------------------------

df_true_local = df_true.copy()
df_true_local["courseId"] = df_true_local["courseId"].astype(str)
df_true_local["comp_id"] = df_true_local["comp_id"].astype(str)

df_compare = df_true_local.merge(
    df_pred_long[["courseId", "comp_id", "pred"]],
    on=["courseId", "comp_id"],
    how="outer"
)

df_compare["real"] = df_compare["real"].fillna(0).astype(int)
df_compare["pred"] = df_compare["pred"].fillna(0).astype(int)

In [ ]:
# -------------------------------------------------------------
# 4) Save datasets: model predictions and comparison
# -------------------------------------------------------------

# model name used to identify the files
model_tag = f"gpt5mini_top{top_k}"

pred_filename = f"label_course_{model_tag}.csv"
df_pred_long.to_csv(pred_filename, index=False)
print(f"Saved model predictions: {pred_filename}")

compare_filename = f"comparison_{model_tag}.csv"
df_compare.to_csv(compare_filename, index=False)
print(f"Saved comparison file: {compare_filename}")

Saved model predictions: label_course_gpt5mini_top10.csv
Saved comparison file: comparison_gpt5mini_top10.csv


In [ ]:
def evaluate_topk(df_compare, top_k):
    """
    df_compare must contain the following columns:
    - courseId
    - comp_id
    - real (0/1)
    - pred (0/1)

    top_k is used only for printing the final output format.
    """

    # -----------------------------
    # F1 (micro e macro)
    # -----------------------------
    f1_micro = f1_score(df_compare["real"], df_compare["pred"], average="micro")
    f1_macro = f1_score(df_compare["real"], df_compare["pred"], average="macro")

    # -----------------------------
    # Hamming Loss
    # -----------------------------
    h_loss = hamming_loss(df_compare["real"], df_compare["pred"])

    # -----------------------------
    # Precision@K
    # -----------------------------
    # precision = correct_predictions / total_predicted (pred=1)
    total_pred = (df_compare["pred"] == 1).sum()
    if total_pred == 0:
        precision_k = 0
    else:
        precision_k = ((df_compare["real"] == 1) & (df_compare["pred"] == 1)).sum() / total_pred

    # -----------------------------
    # Partial Hit@K (course-level hit)
    # -----------------------------
    # A course is considered a partial hit if at least ONE correct competency is predicted among the top K
    course_hits = (
        df_compare
        .groupby("courseId")[["real", "pred"]]
        .apply(lambda g: ((g["real"] == 1) & (g["pred"] == 1)).any())
    )

    parcial_k = course_hits.mean()  # mean = proportion
    
    # -----------------------------
    # Print results in a professional format
    # -----------------------------
    print(f">>> ANALYSIS TOP-{top_k} <<<")
    print("----------------------------------------")
    print(f"Hamming Loss:      {h_loss:.4f}")
    print(f"F1 Score (Micro):  {f1_micro:.4f}")
    print(f"F1 Score (Macro):  {f1_macro:.4f}")
    print("----------------------------------------")
    print(f"Precision@{top_k}:       {precision_k:.4f} (Correct predictions among top {top_k})")
    print(f"Partial Hit@{top_k}:     {parcial_k*100:.2f}% (Courses with at least 1 hit)")
    print("----------------------------------------")

    return {
        "top_k": top_k,
        "precision": precision_k,
        "partial_hit": parcial_k,
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "hamming": h_loss,
    }

## **GPT-4.1-MINI**

### **Top 7**

In [ ]:
evaluate_topk(df_compare, 7)

>>> ANALYSIS TOP-7 <<<
----------------------------------------
Hamming Loss:      0.8471
F1 Score (Micro):  0.1529
F1 Score (Macro):  0.1326
----------------------------------------
Precision@7:       0.1976 (Correct predictions among top 7)
Partial Hit@7:     65.79% (Courses with at least 1 hit)
----------------------------------------


{'top_k': 7,
 'precision': np.float64(0.19764464925755248),
 'partial_hit': np.float64(0.6578947368421053),
 'f1_micro': 0.15287128712871287,
 'f1_macro': 0.1326004809343868,
 'hamming': 0.8471287128712871}

## **GPT-5-MINI**

### **Top 7**

In [23]:
evaluate_topk(df_compare, 7)

>>> ANALYSIS TOP-7 <<<
----------------------------------------
Hamming Loss:      0.8347
F1 Score (Micro):  0.1653
F1 Score (Macro):  0.1418
----------------------------------------
Precision@7:       0.2286 (Correct predictions among top 7)
Partial Hit@7:     59.94% (Courses with at least 1 hit)
----------------------------------------


{'top_k': 7,
 'precision': np.float64(0.22860791826309068),
 'partial_hit': np.float64(0.5994152046783626),
 'f1_micro': 0.16528162511542013,
 'f1_macro': 0.14183835182250396,
 'hamming': 0.8347183748845799}